In [2]:
%matplotlib inline
import os
import sys
from os.path import join as pjoin
from tifffile import imread, imwrite, TiffFile
import numpy as np
import shutil
import matplotlib.pyplot as plt
from glob import glob
import pandas as pd
import cv2
from tqdm import tqdm
import subprocess
from scipy.ndimage import gaussian_filter,median_filter
from scipy.interpolate import interp1d
from scipy.signal import detrend, butter, filtfilt

project_root = '/home/lsh/WF_GoNogo'
if project_root not in sys.path:
    sys.path.append(project_root)
from utils.wfield_utils import *

In [2]:
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
import cupy as cp
cp._default_memory_pool.free_all_blocks()

In [ ]:
# batch_widefield_preproc_with_outlier.py
import os
from os.path import join as pjoin
from glob import glob
import numpy as np
from tifffile import imread, imwrite
import matplotlib.pyplot as plt
import yaml
import time
import re

# ------------------------------
# Funtions
# ------------------------------
def load_config(config_path):
    """load the YAML config"""
    with open(config_path, "r") as f:
        config = yaml.safe_load(f)
    return config

def filename2int(filename):
    nums = re.findall(r'\d+', filename)
    return int(nums[-1]) if nums else -1

# ------------------------------
# Step 1: organize tiffes
# ------------------------------
def organize_tif(folder_path, processPath):
    folder_name = os.path.basename(folder_path)
    tif_path = folder_path + '.tif'
    if os.path.exists(tif_path):
        print(f'importing {tif_path}')
        image_stack = imread(tif_path)
        print(f'finish importing {tif_path}')
    else:
        image_path_ls = glob(os.path.join(folder_path, '*.tif'))
        image_path_ls = sorted(image_path_ls, key=filename2int)
        image_stack = [imread(tiff) for tiff in image_path_ls]
    mean_values = [np.mean(frame) for frame in image_stack]
    output_value = os.path.join(processPath, folder_name + "-Values.csv")
    np.savetxt(output_value, mean_values, delimiter=",")
    return np.array(image_stack)

def fill_missing_frames(folder):
    'Detect and fill missing frames'
    image_paths = sorted(glob(os.path.join(folder, "*.tif")), key=filename2int)
    indices = [filename2int(p) for p in image_paths]
    if not indices:
        print(f"No .tif files found in {folder}")
        return

    all_indices = range(indices[0], indices[-1] + 1)
    missing = sorted(set(all_indices) - set(indices))

    if len(missing) == 0:
        print(f"{folder}: No missing frames")
        return

    print(f"{folder}: Detected {len(missing)} missing frames, filling them...")
    for idx in tqdm(missing):
        prev_path = os.path.join(folder, f"{idx-1}.tif")
        new_path = os.path.join(folder, f"{idx}.tif")
        if os.path.exists(prev_path):
            frame = imread(prev_path)
            imwrite(new_path, frame)
        else:
            print(f"Skipping {idx} (previous frame {idx-1} not found)")
    print(f"{folder}: 补帧完成 ✅")

def merge_two_channels(rawPath, processPath, experiment):
    os.makedirs(os.path.join(processPath, experiment + "-wfield"), exist_ok=True)
    mergePath = os.path.join(processPath, experiment + "-wfield")
    merge_file = os.path.join(mergePath, experiment + "-merged.tif")

    if not os.path.exists(merge_file):
        
        folder_405 = os.path.join(rawPath, experiment + "-405")
        folder_470 = os.path.join(rawPath, experiment + "-470")
        fill_missing_frames(folder_405)
        fill_missing_frames(folder_470)

        tif_405 = organize_tif(folder_405, processPath)
        tif_470 = organize_tif(folder_470, processPath)

        if tif_405.shape[0] != tif_470.shape[0]:
            n_frames = min(tif_405.shape[0], tif_470.shape[0])
            print(f"⚠️ 帧数不一致（405={tif_405.shape[0]}, 470={tif_470.shape[0]}），截取到 {n_frames} 帧")
            tif_405 = tif_405[:n_frames]
            tif_470 = tif_470[:n_frames]

        merged_tif = np.stack([tif_470, tif_405], axis=1)  # shape = (frames, 2, H, W)
        imwrite(merge_file, merged_tif, imagej=True, bigtiff=True)
        merged = imread(merge_file)

        print(f"Merge completed: {merge_file}")
        print(f"Merged frames: {merged.shape[0]}")
    else:
        print(f"Merged {experiment} already exists")

    return merge_file

# ------------------------------
# Step 2: Outlier dection and correction
# ------------------------------
def detect_outlier(mean_values, std_thr=5, qc_path=None, plot=True):
    def find_continuous_outliers(outlier_idx):
        if len(outlier_idx) == 0:
            return []
        segments, start = [], outlier_idx[0]
        for i in range(1, len(outlier_idx)):
            if outlier_idx[i] != outlier_idx[i-1] + 1:
                segments.append((start, outlier_idx[i-1]))
                start = outlier_idx[i]
        segments.append((start, outlier_idx[-1]))
        return segments

    outlier_470 = mean_values[:,0] < mean_values[:,0].mean() - std_thr*mean_values[:,0].std()
    outlier_405 = (mean_values[:,1] < mean_values[:,1].mean() - std_thr*mean_values[:,1].std()) | \
                  (mean_values[:,1] > mean_values[:,1].mean() + std_thr*mean_values[:,1].std())
    outlier_470_segments = find_continuous_outliers(np.where(outlier_470)[0])
    outlier_405_segments = find_continuous_outliers(np.where(outlier_405)[0])

    if plot:
        fig, ax = plt.subplots(figsize=(20,5))
        ax.plot(mean_values[:,0], label='470', color='g')
        ax.plot(mean_values[:,1], label='405', color='purple')
        for s,e in outlier_470_segments:
            ax.plot(np.arange(s,e+1), mean_values[s:e+1,0], 'ko', fillstyle='none')
        for s,e in outlier_405_segments:
            ax.plot(np.arange(s,e+1), mean_values[s:e+1,1], 'ro', fillstyle='none')
        ax.legend()
        plt.title('Raw Mean Values with Outliers')
        
    if qc_path is not None:
            os.makedirs(qc_path, exist_ok=True)
            save_path = os.path.join(qc_path, f"outlier_detection.png")
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            plt.close(fig) 
            print(f"Outlier dection plot saved to: {save_path}")

    return outlier_470_segments, outlier_405_segments

def correct_lum_outlier(merged_tif, outlier_index_470, outlier_index_405, qc_path, plot=True):
    data = imread(merged_tif)  # (nframes, nchannels, H, W)
    print("Merged frames:", data.shape[0])
    for start,end in outlier_index_470:
        data[start:end+1,0,:,:] = 0.5*data[start-1,0,:,:] + 0.5*data[end+1,0,:,:]
    for start,end in outlier_index_405:
        data[start:end+1,1,:,:] = 0.5*data[start-1,1,:,:] + 0.5*data[end+1,1,:,:]

    corrected_file = merged_tif.replace("-merged.tif","-merged-corrected.tif")
    imwrite(corrected_file, data, bigtiff=True)
    print(f"Corrected file saved: {corrected_file}")

    if plot:
        mean_values_corrected = np.stack([data[:,0].mean(axis=(1,2)),
                                          data[:,1].mean(axis=(1,2))], axis=1)
        fig, ax = plt.subplots(figsize=(20,5))
        ax.plot(mean_values_corrected[:,0], label='470', color='g')
        ax.plot(mean_values_corrected[:,1], label='405', color='purple')
        ax.legend(); plt.title('Corrected Mean Values')

    if qc_path is not None:
            os.makedirs(qc_path, exist_ok=True)
            save_path = os.path.join(qc_path, f"outlier_correction.png")
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            plt.close(fig) 
            print(f"Outlier correction plot saved to: {save_path}")

    return corrected_file

# ------------------------------
# Step 3: Downsample
# ------------------------------
def downsample_tif(tif_path, factor=2):
    data = imread(tif_path)
    n_frames = data.shape[0] // factor
    downsampled = data[:n_frames*factor].reshape(n_frames,factor,*data.shape[1:]).mean(axis=1)
    out_path = tif_path.replace(".tif", f"-down{factor}.tif")
    imwrite(out_path, downsampled, bigtiff=True)
    print(f"Downsampled file saved: {out_path}")
    return out_path

# ------------------------------
# Step 4: Crop
# ------------------------------

import numpy as np
import cv2, os
from glob import glob
from tifffile import imread, imwrite
from skimage import measure
import matplotlib.pyplot as plt
from tqdm import tqdm
from os.path import join as pjoin

def show_images(images, titles=None, cmap='hot', colorbar=True, qc_path=None):
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=(12, 4), constrained_layout=True)
    if n == 1:
        axes = [axes]
    ims = []
    for i, img in enumerate(images):
        im = axes[i].imshow(img, cmap=cmap, aspect='equal')
        ims.append(im)
        if titles:
            axes[i].set_title(titles[i])
        axes[i].axis('off')
    if colorbar:
        fig.colorbar(ims[-1], ax=axes, orientation='vertical', fraction=0.02, pad=0.04)

    if qc_path is not None:
        plt.savefig(qc_path, dpi=300, bbox_inches="tight")
        plt.close(fig)
        print(f"Preview figure saved: {qc_path}")
    else:
        plt.show()


def generate_crop_box(processPath, experiment, std_level=80, pad=20, n_preview=600, plot_test=False, qc_path=None
                      ):
    tiff_list_470 = glob(pjoin(processPath, experiment + '*-470/*.tif'))[:n_preview]
    tiff_list_405 = glob(pjoin(processPath, experiment + '*-405/*.tif'))[:n_preview]

    preview_stack_470 = np.array([imread(t) for t in tiff_list_470])
    preview_stack_405 = np.array([imread(t) for t in tiff_list_405])

    preview_std_470 = np.std(preview_stack_470, axis=0)
    preview_std_405 = np.std(preview_stack_405, axis=0)

    if plot_test:
        images = [preview_std_470, preview_std_405, preview_std_470 - preview_std_405]
        title = ['The std of preview stack 470.', 
                'The std of preview stack 405.', 
                'The std of preview stack 470-405.']
        save_path = os.path.join(qc_path, f"std_preview.png")
        show_images(images, titles=title, colorbar=True, qc_path=save_path)
        

    # contour
    contour = measure.find_contours(image=preview_std_470, level=std_level)
    contour_len = [len(_contour) for _contour in contour]
    contour_max = contour[np.argmax(contour_len)]

    # mask
    mask = np.zeros_like(preview_std_470)
    mask_edge = np.flip(contour_max, axis=1).astype(int)
    mask = cv2.fillPoly(mask, [mask_edge], 1)

    # crop box
    ys, xs = np.nonzero(mask)
    top, bottom = ys.min(), ys.max()
    left, right = xs.min(), xs.max()
    top = max(0, top - pad)
    bottom = min(mask.shape[0], bottom + pad)
    left = max(0, left - pad)
    right = min(mask.shape[1], right + pad)
    crop_box = (top, bottom, left, right)

    # visualize
    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    ax[0].imshow(preview_std_470, cmap='hot')
    ax[0].plot(contour_max[:, 1], contour_max[:, 0], 'b', linewidth=1)
    ax[1].imshow(preview_std_405, cmap='hot')
    ax[1].plot(contour_max[:, 1], contour_max[:, 0], 'b', linewidth=1)
    ax[2].imshow(mask, cmap='gray')
    for a in ax:
        a.grid(False)
    # plt.show()
    
    if qc_path:
        save_path2 = os.path.join(qc_path, f"mask_contour.png")
        plt.savefig(save_path2, dpi=300, bbox_inches="tight")
        plt.close(fig)
        print(f"Mask and contour figure saved: {save_path2}")

    print(f"Crop box for {experiment}: {crop_box}")
    return crop_box, mask


def crop_and_save_tif(tif_path, crop_box, chunk_size=256):
    
    data = imread(tif_path).astype(np.uint16)
    n_frames, n_channels, H, W = data.shape
    t, b, l, r = crop_box
    cropped = data[:, :, t:b, l:r]

    # save tif
    out_tif = tif_path.replace(".tif", "-crop.tif")
    imwrite(out_tif, cropped, bigtiff=True)
    print(f"Cropped TIFF saved: {out_tif}")

    # save bin 
    Hc, Wc = cropped.shape[2:]
    out_bin = os.path.join(os.path.dirname(tif_path), f"{n_frames}_{n_channels}_{Hc}_{Wc}_crop_uint16.bin")
    # out_bin = tif_path.replace(".tif", f"-{n_frames}_{n_channels}_{Hc}_{Wc}_crop_uint16.bin")
    with open(out_bin, "wb") as fout:
        for start in tqdm(range(0, n_frames, chunk_size), desc="Writing bin chunks"):
            end = min(start + chunk_size, n_frames)
            cropped[start:end].astype(np.uint16).tofile(fout)
    print(f"Binary file saved: {out_bin}")
    return out_tif, out_bin


# ------------------------------
# Step 5: SVD
# ------------------------------

from scipy.interpolate import interp1d
from scipy.signal import detrend, butter, filtfilt
import os

def mmap_dat(filename,
             mode = 'r',
             nframes = None,
             shape = None,
             dtype='uint16'):
    '''
    Loads frames from a binary file as a memory map.
    This is useful when the data does not fit to memory.
    
    Inputs:
        filename (str)       : fileformat convention, file ends in _NCHANNELS_H_W_DTYPE.dat
        mode (str)           : memory map access mode (default 'r')
                'r'   | Open existing file for reading only.
                'r+'  | Open existing file for reading and writing.                 
        nframes (int)        : number of frames to read (default is None: the entire file)
        offset (int)         : offset frame number (default 0)
        shape (list|tuple)   : dimensions (NCHANNELS, HEIGHT, WIDTH) default is None
        dtype (str)          : datatype (default uint16) 
    Returns:
        A memory mapped  array with size (NFRAMES,NCHANNELS, HEIGHT, WIDTH).

    Example:
        dat = mmap_dat(filename)
    '''
    
    if not os.path.isfile(filename):
        raise OSError('File {0} not found.'.format(filename))
    if shape is None or dtype is None: # try to get it from the filename
        dtype,shape,_ = _parse_binary_fname(filename,
                                            shape = shape,
                                            dtype = dtype)
    if type(dtype) is str:
        dt = np.dtype(dtype)
    else:
        dt = dtype
    if nframes is None:
        # Get the number of samples from the file size
        nframes = int(os.path.getsize(filename)/(np.prod(shape)*dt.itemsize))
    dt = np.dtype(dtype)
    return np.memmap(filename,
                     mode=mode,
                     dtype=dt,
                     shape = (int(nframes),*shape))

def _parse_binary_fname(fname,lastidx=None, dtype = 'uint16', shape = None, sep = '_'):
    '''
    Gets the data type and the shape from the filename 
    This is a helper function to use in load_dat.
    
    out = _parse_binary_fname(fname)
    
    With out default to: 
        out = dict(dtype=dtype, shape = shape, fnum = None)
    '''
    fn = os.path.splitext(os.path.basename(fname))[0]
    fnsplit = fn.split(sep)
    fnum = None
    if lastidx is None:
        # find the datatype first (that is the first dtype string from last)
        lastidx = -1
        idx = np.where([not f.isnumeric() for f in fnsplit])[0]
        for i in idx[::-1]:
            try:
                dtype = np.dtype(fnsplit[i])
                lastidx = i
            except TypeError:
                pass
    if dtype is None:
        dtype = np.dtype(fnsplit[lastidx])
    # further split in those before and after lastidx
    before = [f for f in fnsplit[:lastidx] if f.isdigit()]
    after = [f for f in fnsplit[lastidx:] if f.isdigit()]
    if shape is None:
        # then the shape are the last 3
        shape = [int(t) for t in before[-3:]]
    if len(after)>0:
        fnum = [int(t) for t in after]
    return dtype,shape,fnum


def run_svd_pipeline(processPath, experiment, crop_file, plot_test=False,qc_path=None):
    """
    processPath: session process folder
    experiment: experiment prefix (string)
    approximate_svd_fn: callable(dat_memmap, baseline, n_components) -> (U, SVT)
    hemo_corr_fn: callable(U, SVT_470, SVT_405, fs, ...) -> SVTcorr
    """
    mergePath = os.path.join(processPath, experiment + "-wfield")
    if not os.path.isdir(mergePath):
        raise FileNotFoundError(f"{mergePath} not found")

    dat = mmap_dat(crop_file)
    image_base_min = np.min(dat[-300:, :, :, :], axis=0)
    np.save(pjoin(mergePath, 'frames_average.npy'), image_base_min)

    if plot_test:
        images = [image_base_min[0], image_base_min[1]]
        title = ['The baseline image of 405 channel.', 'The baseline image of 470 channel.']
        show_images(images, titles=title, colorbar=True, qc_path=os.path.join(mergePath, "baseline_images.png"))

    U, SVT = approximate_svd(dat, image_base_min)

    # np.save(pjoin(mergePath,'U.npy'), U)
    # np.save(pjoin(mergePath,'SVT.npy'), SVT)

    if plot_test:
        ncomponents = 10
        ncols = 5
        nrows = int(np.ceil(ncomponents/ncols))
        fig = plt.figure(figsize=[2*ncols, 2*nrows])
        for icomponent in range(ncomponents):
            fig.add_subplot(nrows, ncols, icomponent+1)
            plt.imshow(U[:, :, icomponent], clim=[-0.01,0.01],cmap='Spectral_r')
            plt.title('Component {}'.format(icomponent))
            plt.axis('off')
        # plt.show()
    
    if qc_path:
        save_path = os.path.join(qc_path, f"SVD_components.png")
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        plt.close(fig)
        print(f"Mask and contour figure saved: {save_path}")

    tstart = time.time()
    fs = 10
    freq_highpass = 0.001
    SVT_470 = SVT[:,0::2]
    t = np.arange(SVT.shape[1]) # interpolate the violet
    from scipy.interpolate import interp1d
    SVT_405 = interp1d(t[1::2], SVT[:,1::2], axis=1,
                        fill_value='extrapolate')(t[0::2])
    SVTcorr, rcoeffs, T = hemodynamic_correction(U, 
                                                 SVT_470, 
                                                 SVT_405, 
                                                 fs=fs,
                                                 freq_highpass=freq_highpass,
                                                 freq_lowpass=10)  

    print('Done hemodynamic correction in {0} s '.format(time.time()-tstart))
    # SVTcorr = SVT_470 - SVT_405

    # np.save(pjoin(path,'rcoeffs.npy'), rcoeffs)
    # np.save(pjoin(path,'T.npy'),T)

    from scipy.signal import detrend, butter, filtfilt

    fs = 10  # 帧率 Hz
    highpass = 0.01  # Hz

    # detrend
    SVT_detrended = detrend(SVTcorr, axis=1)

    # highpass filter
    b, a = butter(2, highpass/(fs/2), btype='high')
    SVT_filtered = filtfilt(b, a, SVT_detrended, axis=1)

    np.save(pjoin(mergePath,'SVT_filtered.npy'), SVT_filtered)
    np.save(pjoin(mergePath,'U.npy'), U)
    return U, SVT_filtered


In [4]:
def preprocess_session(config, session):
    print(f"Processing session: {session}")
    rawPath = config["paths"]["raw"].format(
        base_dir=config["base_dir"], mouse_id=config["mouse_id"], session=session
    )
    processPath = config["paths"]["process"].format(
        base_dir=config["base_dir"], mouse_id=config["mouse_id"], session=session
    )
    os.makedirs(processPath, exist_ok=True)
    qc_path = config["paths"]["qc"].format(
        base_dir=config["base_dir"], mouse_id=config["mouse_id"], session=session
    )
    os.makedirs(qc_path, exist_ok=True)

    items = glob(pjoin(rawPath, '202?????-??????-4*'))
    experiments = list(set([os.path.basename(item)[:15] for item in items]))
    print("Experiments found:", experiments)

    for experiment in experiments:
        # 1. Merge two channels
        merged_file = merge_two_channels(rawPath, processPath, experiment)

        # 2. Outlier detection and correction
        values_470 = np.loadtxt(pjoin(processPath, f"{experiment}-470-Values.csv"), delimiter=',')
        values_405 = np.loadtxt(pjoin(processPath, f"{experiment}-405-Values.csv"), delimiter=',')
        n_frames = min(len(values_470), len(values_405))
        mean_values = np.stack([values_470[:n_frames], values_405[:n_frames]], axis=1)
        out470, out405 = detect_outlier(mean_values, std_thr=config["preprocess"]["outlier_std"], qc_path=qc_path, plot=True)
        # out470, out405 = detect_outlier(mean_values, std_thr=2, plot=True)
        corrected_file = correct_lum_outlier(merged_file, out470, out405, plot=True, qc_path=qc_path)
        print("Outlier removal done")

        # 3. Downsample
        file_for_crop = corrected_file
        if config["preprocess"]["downsample"] > 1:
            file_for_crop = downsample_tif(corrected_file, factor=config["preprocess"]["downsample"])
            print(f"Downsampled to: {file_for_crop}")

        # 4. Crop
        crop_box, mask = generate_crop_box(rawPath, experiment, std_level=config["preprocess"]["std_level"], pad=20, n_preview=600,plot_test=True, qc_path=qc_path)
        crop_tiff, crop_file = crop_and_save_tif(file_for_crop, crop_box, chunk_size=config["preprocess"]["chunk_size"])

        # 5. SVD
        run_svd_pipeline(processPath, experiment, crop_file, plot_test=True, qc_path=qc_path)


In [5]:
config_path = "/home/lsh/WF_GoNogo/config/N091_config.yaml"
def preprocess_mice(config_path):
    config = load_config(config_path)
    for session in config["sessions"]:
        preprocess_session(config, session)
    print(f"all done")

In [26]:
preprocess_mice(config_path)

Processing session: 20250926
Experiments found: ['20250926-145141']
/home/lsh/Data_attention/Transfer learning/DATA_linshu/N091/20250926/20250926-145141-405: No missing frames
/home/lsh/Data_attention/Transfer learning/DATA_linshu/N091/20250926/20250926-145141-470: No missing frames
⚠️ 帧数不一致（405=24455, 470=24454），截取到 24454 帧


/home/lsh/WF_GoNogo/wf_gonogo/lib/python3.11/site-packages/tifffile/tifffile.py:1750: UserWarning: <tifffile.TiffWriter '20250926-145141-merged.tif'> writing nonconformant BigTIFF ImageJ
  warnings.warn(


Merge completed: /home/lsh/Data_attention/Transfer learning/DATA_linshu/N091/20250926/process/20250926-145141-wfield/20250926-145141-merged.tif
Merged frames: 24454
Outlier dection plot saved to: /home/lsh/Data_attention/Transfer learning/DATA_linshu/N091/20250926/QualityControl/outlier_detection.png
Merged frames: 24454
Corrected file saved: /home/lsh/Data_attention/Transfer learning/DATA_linshu/N091/20250926/process/20250926-145141-wfield/20250926-145141-merged-corrected.tif
Outlier correction plot saved to: /home/lsh/Data_attention/Transfer learning/DATA_linshu/N091/20250926/QualityControl/outlier_correction.png
Outlier removal done
Downsampled file saved: /home/lsh/Data_attention/Transfer learning/DATA_linshu/N091/20250926/process/20250926-145141-wfield/20250926-145141-merged-corrected-down2.tif
Downsampled to: /home/lsh/Data_attention/Transfer learning/DATA_linshu/N091/20250926/process/20250926-145141-wfield/20250926-145141-merged-corrected-down2.tif
Preview figure saved: /home/ls

Writing bin chunks: 100%|██████████| 48/48 [00:14<00:00,  3.39it/s]

Binary file saved: /home/lsh/Data_attention/Transfer learning/DATA_linshu/N091/20250926/process/20250926-145141-wfield/12227_2_374_442_crop_uint16.bin


...

Writing bin chunks: 100%|██████████| 68/68 [00:27<00:00,  2.50it/s]

Binary file saved: /home/lsh/Data_attention/Transfer learning/DATA_linshu/N091/20250929/process/20250929-180117-wfield/17332_2_461_487_crop_uint16.bin


Preview figure saved: /home/lsh/Data_attention/Transfer learning/DATA_linshu/N091/20250929/process/20250929-180117-wfield/baseline_images.png


Computing SVT from the raw data: 100%|██████████| 35/35 [00:24<00:00,  1.40it/s]


Mask and contour figure saved: /home/lsh/Data_attention/Transfer learning/DATA_linshu/N091/20250929/QualityControl/SVD_components.png
Skipping lowpass on the violet channel.
Done hemodynamic correction in 21.81592893600464 s 
all done
